## 1. Introducción

Esta libreta simula la **llegada incremental de interacciones, etiquetas y datos de uso** al sistema de producción, replicando el comportamiento de un entorno real donde los datos llegan de forma continua desde sistemas externos.

El mecanismo consiste en leer los archivos `data.json` almacenados en la carpeta **`source_buffer`** bajo el volumen **`landing_zone`**, y escribir sus registros en la estructura de directorios de **`events`**, respetando la **partición por año y mes**. Cada ejecución se escribe como un fichero independiente, de forma que el **`Auto Loader`** detecta cada fichero nuevo y lo ingiere de forma incremental sin necesidad de modificar ficheros existentes.

Las interacciones se inyectan en **orden cronológico estricto**. Para cada ventana temporal inyectada:
- Se copian las **etiquetas** cuyo `label_available_date` cae dentro de esa misma ventana.
- Se copian los registros de **usage** cuyo período mensual (`year_month`) ha concluido antes del final de la ventana inyectada.

Labels y usage se procesan **enteramente en Spark** sin bajar a Pandas, dado el volumen de datos.

La simulación es **idempotente**: antes de comenzar, localiza automáticamente el **`timestamp` máximo** ya presente en `events/interactions` y omite todas las filas anteriores o iguales a ese valor.

## 2. Importaciones y configuración

In [ ]:
exec(open("07_Utils.py").read(), globals())

In [ ]:
import json
from calendar import monthrange
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

In [ ]:
# Base paths on the volume
landing_zone_path = Path("/") / "Volumes" / catalog / database / "landing_zone"

source_buffer_int_path   = landing_zone_path / "source_buffer" / "interactions"
source_buffer_lbl_path   = landing_zone_path / "source_buffer" / "labels"
source_buffer_usage_path = landing_zone_path / "source_buffer" / "usage"

events_int_path   = landing_zone_path / "events" / "interactions"
events_lbl_path   = landing_zone_path / "events" / "labels"
events_usage_path = landing_zone_path / "events" / "usage"

# Hours of interaction data to inject in this run
dbutils.widgets.text("hours_to_inject", "3")
hours_to_inject = int(dbutils.widgets.get("hours_to_inject"))

print(f"Source buffer (interactions) : {source_buffer_int_path}")
print(f"Source buffer (labels)       : {source_buffer_lbl_path}")
print(f"Source buffer (usage)        : {source_buffer_usage_path}")
print(f"Events (interactions)        : {events_int_path}")
print(f"Events (labels)              : {events_lbl_path}")
print(f"Events (usage)               : {events_usage_path}")
print(f"Hours to inject              : {hours_to_inject}")

## 3. Punto de continuación

Antes de comenzar la simulación se determina desde qué punto debe continuar, consultando el `timestamp` máximo ya presente en `events/interactions` mediante una lectura distribuida con **Spark**. Todas las interacciones del *buffer* con `timestamp` estrictamente posterior a ese valor serán las candidatas a copiar.

In [ ]:
def _find_latest_events_timestamp():
    """
    Query the maximum `timestamp` already present in `events/interactions`
    and return it as a `datetime` object.
    Returns `None` when no interactions have been copied yet.
    """
    try:
        max_ts = (
            spark.read
                 .json(str(events_int_path / "*" / "*" / "*.json"))
                 .agg(F.max("timestamp"))
                 .first()[0]
        )
        return datetime.fromisoformat(max_ts) if max_ts else None
    except Exception:
        return None


resume_from = _find_latest_events_timestamp()

if resume_from is None:
    print("No prior interactions found. Simulation will start from the beginning of the buffer.")
else:
    print(f"Resuming from timestamp: {resume_from.isoformat()}")
    print("Only interactions strictly after this timestamp will be copied.")

## 4. Carga y ordenación del *buffer* de interacciones

Se leen todos los archivos `data.json` de `source_buffer/interactions` mediante una lectura distribuida con **Spark**, se fusionan en una única lista ordenada cronológicamente por `timestamp` y se filtran las filas ya procesadas según el punto de continuación.

Interacciones se carga en memoria (Pandas) porque se necesita el cursor de continuación y la lógica de ventana temporal para determinar qué inyectar en cada ejecución. Labels y usage se procesan enteramente en Spark.

In [ ]:
def _load_buffer(base_path):
    """
    Extract `_year` and `_month` from the partition path of each record
    and return the full buffer as a flat list of dicts for in-memory processing.
    """
    return (
        spark.read
             .json(str(base_path / "*" / "*" / "data.json"))
             .withColumn("_year",  F.element_at(F.split(F.col("_metadata.file_path"), "/"), -3))
             .withColumn("_month", F.element_at(F.split(F.col("_metadata.file_path"), "/"), -2))
             .toPandas()
             .to_dict("records")
    )


all_int = _load_buffer(source_buffer_int_path)
for row in all_int:
    row["_ts"] = datetime.fromisoformat(str(row["timestamp"]))
all_int.sort(key=lambda row: row["_ts"])

ts_min = all_int[0]["_ts"]
ts_max = all_int[-1]["_ts"]

print(f"Total rows in buffer: {len(all_int):,}")
print(f"Buffer range        : {ts_min}  →  {ts_max}")
print(f"resume_from         : {resume_from}")

if resume_from is not None and resume_from >= ts_max:
    print("⚠️  resume_from is beyond the buffer range. Nothing to inject.")
    pending_int = []

elif resume_from is not None and resume_from < ts_min:
    print(f"⚠️  Gap detected — jumping to buffer start.")
    cutoff      = ts_min + timedelta(hours=hours_to_inject)
    pending_int = [row for row in all_int if row["_ts"] <= cutoff]

elif resume_from is not None:
    cutoff      = resume_from + timedelta(hours=hours_to_inject)
    pending_int = [row for row in all_int if resume_from < row["_ts"] <= cutoff]

else:
    cutoff      = ts_min + timedelta(hours=hours_to_inject)
    pending_int = [row for row in all_int if row["_ts"] <= cutoff]

print(f"cutoff                       : {cutoff if pending_int else 'N/A'}")
print(f"Rows selected for injection  : {len(pending_int):,}")
print(f"Rows skipped (past or future): {len(all_int) - len(pending_int):,}")

## 5. Inyección de datos

Cada ejecución inyecta:
1. Todas las **interacciones** cuyo `timestamp` cae dentro de la ventana `hours_to_inject` horas desde el punto de continuación.
2. Las **etiquetas** cuyo `label_available_date` cae dentro de esa misma ventana — procesadas enteramente en Spark.
3. Los registros de **usage** cuyo período mensual ha concluido antes del último timestamp inyectado — procesados enteramente en Spark.

Todos los tipos se particionan por `(year, month)` y se escriben como ficheros independientes para que el **Auto Loader** los detecte e ingiera de forma incremental.

In [ ]:
def _write_json(dest_path, records):
    """
    Write `records` to `dest_path` on the volume in newline-delimited
    JSON format (one record per line).
    """
    lines = "\n".join(json.dumps(record, default=str) for record in records)
    dbutils.fs.put(dest_path, lines, overwrite=True)


def _clean_row(row):
    """
    Remove internal metadata keys added during loading before writing to disk.
    """
    return {key: value for key, value in row.items() if not key.startswith("_")}

### 5.1. Copia de etiquetas (Spark)

Se filtran las etiquetas cuyo `label_available_date` cae dentro de la ventana temporal. Se hace un `left_anti join` contra `events/labels` para excluir registros ya inyectados. Todo el proceso ocurre en Spark sin bajar a Pandas.

In [ ]:
def _copy_labels_spark(window_start, window_end, batch_timestamp):
    """
    Copy all labels whose `label_available_date` falls within
    [window_start, window_end] and have not yet been injected.
    Deduplication via left_anti join against events/labels.
    Returns the number of labels written.
    """
    # Load already-injected keys — force evaluation inside try to catch PATH_NOT_FOUND
    try:
        injected_df = spark.read.json(str(events_lbl_path / "*" / "*" / "*.json"))
        injected_df.limit(1).count()  # forces path evaluation inside try
        injected = injected_df.select("customer_id", "year_month").distinct()
    except Exception:
        injected = spark.createDataFrame([], "customer_id string, year_month string")

    # Filter buffer by window and anti-join against already injected
    to_copy = (
        spark.read
             .json(str(source_buffer_lbl_path / "*" / "*" / "data.json"))
             .filter(F.col("label_available_date").isNotNull())
             .filter(F.col("label_available_date").cast("timestamp") >= F.lit(window_start))
             .filter(F.col("label_available_date").cast("timestamp") <= F.lit(window_end))
             .join(injected, on=["customer_id", "year_month"], how="left_anti")
    )

    n = to_copy.count()
    if n == 0:
        return 0

    (
        to_copy
        .withColumn("_year",  F.year(F.col("label_available_date").cast("timestamp")).cast("string"))
        .withColumn("_month", F.lpad(F.month(F.col("label_available_date").cast("timestamp")).cast("string"), 2, "0"))
        .write
        .partitionBy("_year", "_month")
        .mode("append")
        .json(str(events_lbl_path))
    )

    return n

### 5.2. Copia de usage (Spark)

Se inyectan los registros de uso cuyo mes de facturación (`year_month`) ha concluido antes del último timestamp inyectado. Se usa `F.last_day` para calcular el fin del período y un `left_anti join` para la deduplicación. Todo el proceso ocurre en Spark sin bajar a Pandas.

In [ ]:
def _copy_usage_spark(window_end, batch_timestamp):
    """
    Copy usage records whose billing period ended on or before `window_end`
    and have not yet been injected.
    Deduplication via left_anti join against events/usage.
    Returns the number of usage records written.
    """
    # Load already-injected keys
    try:
        injected_df = spark.read.json(str(events_usage_path / "*" / "*" / "*.json"))
        injected_df.limit(1).count()  # ← idem
        injected = injected_df.select("customer_id", "year_month").distinct()
    except Exception:
        injected = spark.createDataFrame([], "customer_id string, year_month string")

    # Filter by period end and anti-join against already injected
    to_copy = (
        spark.read
             .json(str(source_buffer_usage_path / "*" / "*" / "data.json"))
             .withColumn(
                 "_period_end",
                 F.last_day(F.to_date(F.col("year_month"), "yyyy-MM"))
             )
             .filter(F.col("_period_end") <= F.lit(window_end.date()))
             .drop("_period_end")
             .join(injected, on=["customer_id", "year_month"], how="left_anti")
    )

    n = to_copy.count()
    if n == 0:
        return 0

    (
        to_copy
        .withColumn("_year",  F.col("year_month").substr(1, 4))
        .withColumn("_month", F.col("year_month").substr(6, 2))
        .write
        .partitionBy("_year", "_month")
        .mode("append")
        .json(str(events_usage_path))
    )

    return n

### 5.3. Ejecución

Las interacciones se agrupan por partición `(year, month)` y se escriben en `events/interactions` como un fichero `.json` independiente por ejecución. A continuación se inyectan labels y usage para la misma ventana temporal.

In [ ]:
if not pending_int:
    print("No pending interactions for this time window.")
else:
    n_pending = len(pending_int)
    batch_ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    header    = (
        f"Injecting {n_pending:,} interactions covering {hours_to_inject} hour(s) "
        f"from {pending_int[0]['_ts']} to {pending_int[-1]['_ts']}."
    )
    separator = "-" * len(header)

    print(header)
    print(separator)

    # 1. Copy interactions grouped by (year, month) partition
    int_partitions = {}
    for row in pending_int:
        key = (row["_year"], row["_month"])
        int_partitions.setdefault(key, []).append(row)

    for (year, month), rows in int_partitions.items():
        dest_path = str(events_int_path / year / month / f"batch_{batch_ts}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])

    # 2. Copy labels whose available date falls within the injection window
    n_lbl = _copy_labels_spark(pending_int[0]["_ts"], pending_int[-1]["_ts"], batch_ts)

    # 3. Copy usage records whose monthly period ended before window_end
    n_usage = _copy_usage_spark(pending_int[-1]["_ts"], batch_ts)

    print(separator)
    print("Injection complete.")
    print(f"  Interactions injected : {n_pending:,}")
    print(f"  Labels injected       : {n_lbl:,}")
    print(f"  Usage records injected: {n_usage:,}")

## 6. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Continuación idempotente**: Consulta el `timestamp` máximo ya presente en `events/interactions` y omite todas las filas anteriores, evitando duplicados en ejecuciones repetidas. Si hay un gap entre el último timestamp inyectado y el inicio del buffer, salta automáticamente al inicio del buffer.
2. **Inyección por ventana temporal**: Las interacciones se procesan en orden cronológico estricto y se escriben en `events/interactions` respetando la partición `year/month`. El parámetro `hours_to_inject` controla cuántas horas de datos se inyectan por ejecución.
3. **Labels en Spark**: Las etiquetas se filtran, deduplicан y escriben enteramente en Spark mediante `left_anti join` contra `events/labels`, sin bajar a Pandas.
4. **Usage en Spark**: Los registros de uso se filtran por fin de período mensual (`last_day(year_month)`), se deduplicан via `left_anti join` contra `events/usage` y se escriben particionados por año y mes.

### Arquitectura de procesamiento

| Entidad | Carga | Filtrado | Escritura |
|---|---|---|---|
| interactions | Spark → Pandas | en memoria | `dbutils.fs.put` |
| labels | Spark | Spark | Spark `.write.json` |
| usage | Spark | Spark | Spark `.write.json` |

### ¿Cuándo ejecutar esta libreta?

Durante las primeras pruebas se ejecuta manualmente para validar el flujo completo. Una vez validado, se integra como tarea `Run_Simulation` en el trabajo **Simulation Pipeline**, programado cada hora para inyectar automáticamente nuevas instancias antes de que el pipeline Medallion las ingiera.